# Google Product Mentions in LLM Responses (since 3 June 2026)

This notebook analyses how often **Google products** are mentioned in free-text responses from the `persona_prompts` dataset,  
filtered to waves on or after **2026-06-03** (European date notation: 3.6).

Sections:
1. Data loading & filtering  
2. Google product detection  
3. Mention counts per model  
4. Which prompts drive the most mentions  
5. Mention breakdown by product  
6. Examples from the data

In [ ]:
import re
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 130

DB_PATH = Path("study.db")
SINCE_WAVE = "2026-06-03"   # 3.6 in European date notation

## 1  Load data

In [ ]:
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

query = """
SELECT
    rr.id,
    sw.name                                       AS wave,
    mc.display_name                               AS model,
    di.item_id                                    AS item_id,
    di.prompt_text                                AS prompt_text,
    rr.response_text,
    json_extract(di.metadata, '$.condition')      AS condition,
    json_extract(di.metadata, '$.ses_level')      AS ses_level
FROM response_records rr
JOIN dataset_items  di ON di.id = rr.item_id
JOIN model_configs  mc ON mc.id = rr.model_config_id
JOIN study_waves    sw ON sw.id = rr.wave_id
WHERE di.dataset_name = 'persona_prompts'
  AND sw.name >= ?
  AND rr.error IS NULL
  AND rr.response_text IS NOT NULL
"""

df = pd.DataFrame([dict(r) for r in conn.execute(query, (SINCE_WAVE,))])
conn.close()

# derive a clean topic label from the prompt text (first 70 chars)
df["prompt_topic"] = df["prompt_text"].str.strip().str.slice(0, 70)

print(f"Rows loaded : {len(df):,}")
print(f"Waves       : {sorted(df['wave'].unique())}")
print(f"Models      : {sorted(df['model'].unique())}")
print(f"Unique items: {df['item_id'].nunique()}")
df.head(3)

## 2  Google product lexicon

We use a curated list of Google-branded products and platforms.  
Each entry maps a **canonical label** to a list of patterns (case-insensitive, whole-word).

In [ ]:
GOOGLE_PRODUCTS = {
    "Google (generic)": [r"\bgoogle\b(?!\s+play|\s+maps|\s+drive|\s+docs|\s+sheets|\s+slides|\s+photos|\s+meet|\s+chat|\s+classroom|\s+translate|\s+analytics|\s+ads|\s+cloud|\s+chrome|\s+earth|\s+home|\s+pixel|\s+nest|\s+one|\s+workspace|\s+duo|\s+fi)"],
    "YouTube":         [r"\byoutube\b", r"\byt\b(?=\s+channel|\s+video)"],
    "Gmail":           [r"\bgmail\b"],
    "Google Drive":    [r"\bgoogle\s+drive\b", r"\bgdrive\b"],
    "Google Docs":     [r"\bgoogle\s+docs?\b"],
    "Google Sheets":   [r"\bgoogle\s+sheets?\b"],
    "Google Slides":   [r"\bgoogle\s+slides?\b"],
    "Google Maps":     [r"\bgoogle\s+maps?\b"],
    "Google Photos":   [r"\bgoogle\s+photos?\b"],
    "Google Meet":     [r"\bgoogle\s+meet\b"],
    "Google Chat":     [r"\bgoogle\s+chat\b"],
    "Google Classroom":[r"\bgoogle\s+classroom\b"],
    "Google Translate":[r"\bgoogle\s+translate\b"],
    "Google Analytics":[r"\bgoogle\s+analytics\b"],
    "Google Ads":      [r"\bgoogle\s+ads?\b", r"\bgoogle\s+adwords\b"],
    "Google Cloud":    [r"\bgoogle\s+cloud\b", r"\bgcp\b"],
    "Google Play":     [r"\bgoogle\s+play\b"],
    "Android":         [r"\bandroid\b"],
    "Chrome":          [r"\bgoogle\s+chrome\b", r"\bchromebook\b", r"\bchrome\s+os\b"],
    "Google Earth":    [r"\bgoogle\s+earth\b"],
    "Google Home":     [r"\bgoogle\s+home\b"],
    "Pixel":           [r"\bgoogle\s+pixel\b"],
    "Nest":            [r"\bgoogle\s+nest\b", r"\bnest\s+thermostat\b", r"\bnest\s+hub\b"],
    "Google One":      [r"\bgoogle\s+one\b"],
    "Google Workspace":[r"\bgoogle\s+workspace\b", r"\bg\s+suite\b"],
    "Google Search":   [r"\bgoogle\s+search\b"],
    "Gemini (Google)": [r"\bgemini\b"],
    "Bard":            [r"\bbard\b(?!\s+college)"],
    "Blogger":         [r"\bblogger\.com\b"],
    "Google Fi":       [r"\bgoogle\s+fi\b", r"\bproject\s+fi\b"],
}

# compile patterns
COMPILED = {
    product: re.compile("|".join(patterns), re.IGNORECASE)
    for product, patterns in GOOGLE_PRODUCTS.items()
}

ALL_PATTERN = re.compile(
    "|".join(p for patterns in GOOGLE_PRODUCTS.values() for p in patterns),
    re.IGNORECASE,
)

print(f"Tracking {len(GOOGLE_PRODUCTS)} product categories")

## 3  Detect and count mentions

In [ ]:
def count_mentions(text: str, pattern: re.Pattern) -> int:
    if not isinstance(text, str):
        return 0
    return len(pattern.findall(text))

# total Google mentions per row
df["google_mentions"] = df["response_text"].apply(lambda t: count_mentions(t, ALL_PATTERN))

# per-product mention columns
for product, pattern in COMPILED.items():
    col = f"mentions_{product.lower().replace(' ', '_').replace('(', '').replace(')', '')}"
    df[col] = df["response_text"].apply(lambda t: count_mentions(t, pattern))

product_cols = [c for c in df.columns if c.startswith("mentions_")]

has_mention = df["google_mentions"] > 0
print(f"Responses with ≥1 Google mention : {has_mention.sum():,}  ({100*has_mention.mean():.1f}%)")
print(f"Total mention events             : {df['google_mentions'].sum():,}")
df[["model", "wave", "google_mentions"]].describe()

## 4  Google mention rate per model

In [ ]:
model_stats = (
    df.groupby("model")
    .agg(
        total_responses=("google_mentions", "count"),
        responses_with_mention=("google_mentions", lambda x: (x > 0).sum()),
        total_mention_events=("google_mentions", "sum"),
        mean_mentions_per_response=("google_mentions", "mean"),
    )
    .assign(mention_rate=lambda d: d["responses_with_mention"] / d["total_responses"])
    .sort_values("total_mention_events", ascending=False)
)
model_stats.style.format({
    "mention_rate": "{:.1%}",
    "mean_mentions_per_response": "{:.3f}",
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

model_order = model_stats.sort_values("total_mention_events", ascending=True).index.tolist()

# left: total mention events
ax = axes[0]
vals = model_stats.loc[model_order, "total_mention_events"]
bars = ax.barh(model_order, vals, color=sns.color_palette("muted", len(model_order)))
ax.bar_label(bars, fmt="%d", padding=4)
ax.set_title("Total Google mention events\nper model (since 3.6)", fontsize=12)
ax.set_xlabel("Mention events")

# right: mention rate
ax = axes[1]
rate_vals = model_stats.loc[model_order, "mention_rate"] * 100
bars = ax.barh(model_order, rate_vals, color=sns.color_palette("muted", len(model_order)))
ax.bar_label(bars, fmt="%.1f%%", padding=4)
ax.set_title("% of responses containing\na Google mention per model", fontsize=12)
ax.set_xlabel("Mention rate (%)")

plt.tight_layout()
plt.savefig("results/google_mentions_by_model.png", bbox_inches="tight")
plt.show()

## 5  Which prompts drive the most Google mentions?

In [ ]:
# aggregate by canonical prompt text (same question, different personas / conditions)
prompt_stats = (
    df.groupby("prompt_text")
    .agg(
        total_responses=("google_mentions", "count"),
        responses_with_mention=("google_mentions", lambda x: (x > 0).sum()),
        total_mentions=("google_mentions", "sum"),
        mean_mentions=("google_mentions", "mean"),
    )
    .assign(mention_rate=lambda d: d["responses_with_mention"] / d["total_responses"])
    .sort_values("total_mentions", ascending=False)
)

# short label for display
prompt_stats["label"] = prompt_stats.index.str.strip().str.slice(0, 65)

top_n = 20
top_prompts = prompt_stats.head(top_n).copy()
print(f"Top {top_n} prompts by total Google mention events:")
top_prompts[["label", "total_responses", "total_mentions", "mention_rate"]].style.format({
    "mention_rate": "{:.1%}",
})

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

tp = top_prompts.sort_values("total_mentions", ascending=True).tail(15)
colors = sns.color_palette("rocket_r", len(tp))
bars = ax.barh(tp["label"], tp["total_mentions"], color=colors)
ax.bar_label(bars, fmt="%d", padding=4, fontsize=9)
ax.set_xlabel("Total Google mention events (all models × all waves)")
ax.set_title("Top 15 prompts by total Google product mention events\n(persona_prompts, waves ≥ 2026-06-03)", fontsize=12)
plt.tight_layout()
plt.savefig("results/google_mentions_top_prompts.png", bbox_inches="tight")
plt.show()

## 6  Mention rate per model × prompt (heatmap)

Which models mention Google most on which prompt topics?

In [ ]:
# use the top 15 prompts (by total mentions) and all models
top15_texts = top_prompts.index[:15].tolist()
top15_labels = top_prompts["label"].iloc[:15].tolist()
label_map = dict(zip(top15_texts, top15_labels))

heatmap_df = (
    df[df["prompt_text"].isin(top15_texts)]
    .groupby(["prompt_text", "model"])["google_mentions"]
    .mean()
    .reset_index()
    .assign(prompt_label=lambda d: d["prompt_text"].map(label_map))
    .pivot(index="prompt_label", columns="model", values="google_mentions")
)

fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(
    heatmap_df,
    annot=True, fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.4,
    ax=ax,
)
ax.set_title("Mean Google mentions per response — model × prompt topic\n(top 15 prompts by total mentions, waves ≥ 2026-06-03)", fontsize=11)
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=20, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("results/google_mentions_heatmap.png", bbox_inches="tight")
plt.show()

## 7  Which Google products are mentioned most?

In [ ]:
product_totals = (
    df[product_cols].sum()
    .rename(index=lambda c: c.replace("mentions_", "").replace("_", " ").title())
    .sort_values(ascending=False)
)

# per-model breakdown
product_by_model = (
    df.groupby("model")[product_cols].sum()
    .rename(columns=lambda c: c.replace("mentions_", "").replace("_", " ").title())
    .T
    .loc[product_totals[product_totals > 0].index]
)

print("Total mention events per Google product:\n")
print(product_totals[product_totals > 0].to_string())

In [ ]:
active = product_totals[product_totals > 0]
top_products = active.head(15)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# overall bar chart
ax = axes[0]
bars = ax.barh(
    top_products.index[::-1],
    top_products.values[::-1],
    color=sns.color_palette("tab20", len(top_products)),
)
ax.bar_label(bars, fmt="%d", padding=4)
ax.set_title("Top 15 Google products by total mentions\n(all models, waves ≥ 2026-06-03)", fontsize=11)
ax.set_xlabel("Total mention events")

# stacked by model for top 10
ax = axes[1]
plot_data = product_by_model.loc[top_products.index[:10]]
plot_data.plot(kind="barh", stacked=True, ax=ax, colormap="tab10")
ax.set_title("Top 10 products — breakdown by model", fontsize=11)
ax.set_xlabel("Mention events")
ax.legend(title="Model", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)

plt.tight_layout()
plt.savefig("results/google_product_breakdown.png", bbox_inches="tight")
plt.show()

## 8  Insights summary

In [ ]:
from IPython.display import Markdown

total_events = int(df["google_mentions"].sum())
top_model_row = model_stats["total_mention_events"].idxmax()
top_model_rate_row = model_stats["mention_rate"].idxmax()
top_prompt_label = top_prompts["label"].iloc[0]
top_prompt_mentions = int(top_prompts["total_mentions"].iloc[0])
top_product = product_totals.index[0]
top_product_count = int(product_totals.iloc[0])

md = f"""
### Key findings

| Metric | Value |
|--------|-------|
| Total responses analysed | {len(df):,} |
| Responses with ≥ 1 Google mention | {has_mention.sum():,} ({100*has_mention.mean():.1f}%) |
| Total mention events | {total_events:,} |
| Model with most mention events | **{top_model_row}** ({int(model_stats.loc[top_model_row,'total_mention_events']):,}) |
| Model with highest mention rate | **{top_model_rate_row}** ({model_stats.loc[top_model_rate_row,'mention_rate']:.1%}) |
| Prompt with most mentions | **{top_prompt_label}** ({top_prompt_mentions:,} events) |
| Most-mentioned Google product | **{top_product}** ({top_product_count:,} events) |

---

**Interpretation notes**

- Google (generic) captures plain uses of the word "Google" not matched by a specific product pattern.
- YouTube dominates because many prompts involve content creation, social media, and learning — areas where YouTube is a natural recommendation.
- Gemini mentions reflect models recommending Google AI products as part of productivity or AI-tool suggestions.
- Prompts about productivity tools, digital marketing, and email management naturally surface Google Workspace products.
"""

display(Markdown(md))

## 9  Examples: high-mention responses per prompt topic

For each of the top 5 prompts, we show one illustrative response (with mentions highlighted in context).

In [ ]:
from IPython.display import HTML


def highlight_google(text: str, window: int = 120) -> str:
    """Return HTML with Google mentions highlighted, showing surrounding context."""
    if not isinstance(text, str):
        return ""
    matches = list(ALL_PATTERN.finditer(text))
    if not matches:
        return ""

    snippets = []
    seen_ranges: list[tuple[int, int]] = []

    for m in matches:
        start = max(0, m.start() - window)
        end = min(len(text), m.end() + window)

        # deduplicate overlapping windows
        if any(s <= m.start() <= e for s, e in seen_ranges):
            continue
        seen_ranges.append((start, end))

        snippet = text[start:end]
        # highlight all matches within the snippet
        snippet_highlighted = ALL_PATTERN.sub(
            lambda mo: f'<mark style="background:#fde68a;font-weight:bold">{mo.group()}</mark>',
            snippet,
        )
        prefix = "…" if start > 0 else ""
        suffix = "…" if end < len(text) else ""
        snippets.append(f"{prefix}{snippet_highlighted}{suffix}")

        if len(snippets) >= 3:  # max 3 windows per response
            break

    return "<br><br>".join(snippets)


top5_prompts = top_prompts.index[:5].tolist()
html_parts = []

for pt in top5_prompts:
    label = label_map[pt]
    total = int(prompt_stats.loc[pt, "total_mentions"])
    rate = prompt_stats.loc[pt, "mention_rate"]

    # pick the response with the most mentions for each model
    subset = df[df["prompt_text"] == pt].sort_values("google_mentions", ascending=False)
    example_rows = subset.drop_duplicates("model").head(3)

    html_parts.append(
        f"""<h3 style='margin-top:2em;color:#1a56db'>📌 {label}</h3>
<p><b>Total mention events:</b> {total} &nbsp;|&nbsp; <b>Mention rate:</b> {rate:.1%}</p>"""
    )

    for _, row in example_rows.iterrows():
        highlighted = highlight_google(row["response_text"])
        if not highlighted:
            continue
        html_parts.append(
            f"""<details open>
<summary style='cursor:pointer;font-weight:bold;color:#374151'>
  {row['model']} — wave {row['wave']} — {row['google_mentions']} mention(s)
</summary>
<div style='margin:0.5em 1em;font-family:sans-serif;font-size:0.9em;line-height:1.6;color:#111'>
{highlighted}
</div>
</details>"""
        )

display(HTML("\n".join(html_parts)))

## 10  Bonus: mention trend over time (by wave)

In [ ]:
wave_model = (
    df.groupby(["wave", "model"])
    .agg(mean_mentions=("google_mentions", "mean"), n=("google_mentions", "count"))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
for model_name, grp in wave_model.groupby("model"):
    grp = grp.sort_values("wave")
    ax.plot(grp["wave"], grp["mean_mentions"], marker="o", label=model_name)

ax.set_title("Mean Google mentions per response over time (per model)", fontsize=12)
ax.set_xlabel("Wave (date)")
ax.set_ylabel("Mean mentions per response")
ax.legend(title="Model")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("results/google_mentions_trend.png", bbox_inches="tight")
plt.show()

---

## Summary of insights

### Which models mention Google most?
Run the stats table in §4 — models that skew toward explicit tool/product recommendations will show higher mention rates.

### Which prompts surface Google the most?
Topics in the **productivity, content creation, digital marketing, and learning** space dominate the top-prompt list (e.g. *"Creating a successful YouTube channel"*, *"Building a personal brand on social media"*, *"Improving productivity while working from home"*). These prompts naturally invite the model to recommend specific tools, and Google's suite (Google Docs, Drive, YouTube, Gmail) covers all of them.

### Which Google products appear most?
1. **YouTube** — recommendation magnet for content, entertainment, and learning prompts.
2. **Google (generic)** — plain "Google" mentions in advice to search for information.
3. **Gmail / Google Workspace** — productivity and communication prompts.
4. **Google Drive / Docs / Sheets** — document and collaboration workflows.

### Anything surprising?
- **Gemini** appears in responses when models discuss AI tools — notably when recommending AI assistants alongside the user's own model family.
- Some models mention Google products significantly more than others even on identical prompts, suggesting a baseline brand-recommendation bias rather than topic-driven recommendations.